[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

/home/user/miniconda/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [15]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    # pass  # temperature, top-k filter, top-p filter, sample
    logits = logits / max(temperature, 1e-8)
    if top_k > 0:
        val = logits.topk(top_k)[0][-1]
        logits[logits < val] = float("-inf")

    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        probs = torch.softmax(sorted_logits, dim=-1)
        cumsum = torch.cumsum(probs, dim=-1)
        mask = (cumsum - probs) > top_p
        sorted_logits = sorted_logits.masked_fill(mask, float("-inf"))
        logits = torch.zeros_like(logits).scatter_(0, sorted_idx, sorted_logits)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1).item()

In [16]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: 1
top_p=0.5: 1
temp=0.01: 1


In [17]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (2.7ms)
  ✅ [2/4] Low temperature concentrates (6.7ms)
  ✅ [3/4] All tokens reachable (no filtering) (153.2ms)
  ✅ [4/4] Returns valid index (3.9ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (166.4ms total)
  Progress saved. Run status() to see your dashboard.

